# 01d · CoCiP end to end on STRATUM's own ERA5 files (Phase 1, task 1.4)

Exploration only. Needs the one-day ERA5 test download (15 Jan 2026,
`uv run stratum download-era5 --month 2026-01 --days 15`).

1. A synthetic Delhi → Mumbai flight at FL350 (03:00 UTC) through dry air: no contrail.
2. A synthetic flight through the most ice-supersaturated cold spot at 250 hPa, 06:00 UTC:
   contrails form, persist and are tracked; STRATUM's SAC is cross-checked against
   pycontrails waypoint by waypoint.

Results are written up in `reports/phase1/cocip_end_to_end.md`.

In [ ]:
import json, time, warnings
import numpy as np, pandas as pd, xarray as xr
import pycontrails
from pycontrails import Flight, MetDataset
from pycontrails.models.cocip import Cocip
from pycontrails.models.humidity_scaling import ConstantHumidityScaling
from pycontrails.models.ps_model import PSFlight
from stratum.config import great_circle, load_domain
from stratum.paths import drive_root
warnings.filterwarnings("ignore")

root = drive_root() / "raw/era5"
met = MetDataset(xr.open_zarr(root / "pl/2026-01_days-15-15/data.zarr").load())
rad = MetDataset(xr.open_zarr(root / "sl/2026-01_days-15-15/data.zarr").load())

# Synthetic test flight: VIDP → VABB great circle, FL350, departing 03:00 UTC, 1-minute points.
line = np.asarray(great_circle(load_domain().corridor).coords)
n = 110
idx = np.linspace(0, len(line) - 1, n)
lon = np.interp(idx, np.arange(len(line)), line[:, 0]); lat = np.interp(idx, np.arange(len(line)), line[:, 1])
t = pd.date_range("2026-01-15T03:00", periods=n, freq="1min")
fl = Flight(longitude=lon, latitude=lat, altitude_ft=np.full(n, 35000.0), time=t,
            aircraft_type="A320", flight_id="TEST-DEL-BOM")

t0 = time.time()
cocip = Cocip(met=met, rad=rad, humidity_scaling=ConstantHumidityScaling(rhi_adj=1.0),
              aircraft_performance=PSFlight(), max_age=np.timedelta64(6, "h"))
out = cocip.eval(source=fl)
df = out.dataframe
res = {
    "pycontrails": pycontrails.__version__,
    "runtime_s": round(time.time() - t0, 1),
    "waypoints": len(df),
    "sac_satisfied": int(df["sac"].fillna(0).sum()) if "sac" in df else None,
    "persistent_contrail_waypoints": int((df.get("persistent_1", pd.Series(0)).fillna(0) > 0).sum()),
    "ef_total_J": float(df.get("ef", pd.Series(0.0)).fillna(0).sum()),
    "rhi_range": [float(df["rhi"].min()), float(df["rhi"].max())] if "rhi" in df else None,
    "T_range_K": [float(df["air_temperature"].min()), float(df["air_temperature"].max())],
    "engine_efficiency_range": [float(df["engine_efficiency"].min()), float(df["engine_efficiency"].max())] if "engine_efficiency" in df else None,
    "contrail_evolution_rows": 0 if cocip.contrail is None else len(cocip.contrail),
    "output_columns": sorted(df.columns)[:60],
}
print(json.dumps(res, indent=1))

In [ ]:
import json, warnings
import numpy as np, pandas as pd, xarray as xr
from pycontrails import Flight, MetDataset
from pycontrails.models.cocip import Cocip
from pycontrails.models.humidity_scaling import ConstantHumidityScaling
from pycontrails.models.ps_model import PSFlight
from stratum.paths import drive_root
from stratum.physics.sac import schmidt_appleman
from stratum.physics.thermo import rhi_from_q
from stratum.physics.units import ft_to_pressure
warnings.filterwarnings("ignore")
root = drive_root() / "raw/era5"
pl = xr.open_zarr(root / "pl/2026-01_days-15-15/data.zarr").load()

# Find the most supersaturated cold spot at 250 hPa, 06 UTC, inside the padded box.
s = pl.sel(level=250.0, time="2026-01-15T06:00")
rhi = rhi_from_q(s["specific_humidity"].values, s["air_temperature"].values, 25000.0)
cold = s["air_temperature"].values < 225.0
i, j = np.unravel_index(np.nanargmax(np.where(cold, rhi, -1)), rhi.shape)
lon0, lat0 = float(s.longitude[i]), float(s.latitude[j])
print(f"ISSR target: {lat0:.2f}N {lon0:.2f}E, RHi {rhi[i, j]:.2f}, T {float(s['air_temperature'].values[i, j]):.1f} K")

# Flight crossing it west→east at FL340 (≈ 250 hPa), passing the spot at ~06:00 UTC.
n = 60
lon = np.linspace(lon0 - 2.5, lon0 + 2.5, n); lat = np.full(n, lat0)
t = pd.date_range("2026-01-15T05:30", periods=n, freq="1min")
fl = Flight(longitude=lon, latitude=lat, altitude_ft=np.full(n, 34000.0), time=t, aircraft_type="A320", flight_id="TEST-ISSR")
cocip = Cocip(met=MetDataset(pl), rad=MetDataset(xr.open_zarr(root / "sl/2026-01_days-15-15/data.zarr").load()),
              humidity_scaling=ConstantHumidityScaling(rhi_adj=1.0), aircraft_performance=PSFlight(),
              max_age=np.timedelta64(6, "h"))
df = cocip.eval(source=fl).dataframe

# Our SAC on exactly CoCiP's interpolated inputs.
p = ft_to_pressure(np.full(n, 34000.0))
ours = schmidt_appleman(df["air_temperature"].values, df["rhi"].values, p, df["engine_efficiency"].values)
agree = float(np.mean(ours.forms == (df["sac"].values > 0)))
ef = df["ef"].fillna(0)
print(json.dumps({
    "waypoints": n,
    "rhi_max": round(float(df["rhi"].max()), 3),
    "pycontrails_sac_waypoints": int((df["sac"] > 0).sum()),
    "stratum_sac_forms_waypoints": int(ours.forms.sum()),
    "sac_agreement_fraction": round(agree, 3),
    "persistent_contrail_waypoints": int((df["persistent_1"].fillna(0) > 0).sum()),
    "stratum_persists_waypoints": int(ours.persists.sum()),
    "ef_total_J": f"{ef.sum():.3e}",
    "ef_positive_waypoints": int((ef > 0).sum()),
    "contrail_evolution_rows": 0 if cocip.contrail is None else len(cocip.contrail),
    "max_contrail_age_h": None if cocip.contrail is None else round(float(cocip.contrail["age"].max() / np.timedelta64(1, "h")), 2),
}, indent=1))